# Notebook de entrenamiento de modelos

En este Notebook, se toman los datos crudos del dataset de Tipo de Cubierta Forestal de la base da datos mysql alojados en la tabla **forest_cover_processed** para entrenar los siguientes modelos:

- Support Vector Machine
- Decision Tree
- K-Nearest Neighbours

Estos modelos se usan para predecir a el tipo de cobertura forestal en base a variables cartográficas.

In [1]:
!pip install boto3 sqlalchemy pymysql

### Importación de librerías

In [2]:
import pandas as pd
from sqlalchemy import create_engine
import pymysql
import pandas as pd
import pickle
from pathlib import Path
import json
import os
import boto3

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix


Creación de la carpeta artifacts

In [3]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

### Función para leer tabla de data lista para entrenamiento

In [4]:
def load_processed_table():

    user = "mlops_user"
    password = "mlops_pass"      # cambia si tu docker usa otro
    host = "mysql_db"
    port = 3306
    database = "mlops_db"

    engine = create_engine(
        f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
    )

    df = pd.read_sql_table("covertype_processed", engine)

    return df

In [5]:
df_processed = load_processed_table()


In [6]:
df_processed.head()

,uuid,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,...,Soil_Type_C7755,Soil_Type_C7756,Soil_Type_C7757,Soil_Type_C8771,Soil_Type_C8772,Soil_Type_C8776,Wilderness_Area_Cache,Wilderness_Area_Commanche,Wilderness_Area_Rawah,Cover_Type
0,1465ec2ae8a22e6ce304d5662fc57e0b77def28094c950...,3201,233,5,474,39,6018,212,244,169,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1
1,79a17a87f6291de44f3c724681567479fa99ce828ad0ff...,2964,351,11,60,4,726,199,221,159,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1
2,9ff9ffc91da91064f7cd273940d979829e3dfa9f25480c...,3177,126,14,90,-17,2147,243,229,112,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
3,5e804d3c9993a6c1fd5e13747f78cdf9a6e997eaab61a9...,3241,0,1,42,0,5609,217,236,156,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1
4,754114ae0ec2246217ac8fb0245a4e8c2429bd134ae9c6...,3286,22,11,342,59,3150,213,217,139,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0


In [7]:
len(df_processed.columns)

38

In [8]:
cat_cols = []

for c in df_processed.columns:
    if c.startswith("Soil") or c.startswith("Wilderness") :
        cat_cols.append(c)
        #print(c)
        

In [9]:
num_cols = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]

df_processed[num_cols] = df_processed[num_cols].fillna(df_processed[num_cols].median())
df_processed[cat_cols] = df_processed[cat_cols].fillna(float(0))

### Preprocesamiento de datos

En la siguiente celda, se define la función que preprocesa los datos para pasarlos por los modelos

In [10]:
# ========= Preprocesamiento de datos para el conjunto de datos de pingüinos =========

import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix


def preprocess_data(df):
    

    df = df.copy()

    target = "Cover_Type"

    cat_cols = ["Soil_Type", "Wilderness_Area"]

    num_cols = [
        "Elevation",
        "Aspect",
        "Slope",
        "Horizontal_Distance_To_Hydrology",
        "Vertical_Distance_To_Hydrology",
        "Horizontal_Distance_To_Roadways",
        "Hillshade_9am",
        "Hillshade_Noon",
        "Hillshade_3pm",
        "Horizontal_Distance_To_Fire_Points"
    ]


    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    df[cat_cols] = df[cat_cols].fillna("Unknown")

    # OneHotEncoder para variables categóricas
    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    df_cat = ohe.fit_transform(df[cat_cols])
    cat_feature_names = ohe.get_feature_names_out(cat_cols)
    df_cat_df = pd.DataFrame(df_cat, columns=cat_feature_names, index=df.index)

    # Concatenar numéricas y categóricas
    df_processed = pd.concat([df[num_cols], df_cat_df, df[target]], axis=1)
    

    target = "Cover_Type"

    X = df_processed.drop(columns=target)
    
    y = df[target]

    encoders = {"onehot": ohe}

    return df_processed, X, y, encoders
    #return df, X, y, #encoders



In [11]:
#df = pd.read_csv("dataset_test\dataset_test.csv")

### Definición de funciones para conexión a la BD

En la siguiente celda, se definen las funciones que perimten hacer la conexión con la base de datos, así como escribir y cargar datos a la misma.

## Entrenamiento de modelos

A continaución se definen las funciones para entrenar los modelos de ML.

In [12]:
# ========================= Funciones de entrenamiento para modelos de clasificación =========================

def train_decision_tree(df):
    """Entrena un modelo de Árbol de Decisión usando OneHotEncoder.
    
    Preprocesa los datos, divide en conjuntos de entrenamiento y prueba,
    entrena un DecisionTreeClassifier y evalúa su desempeño.
    
    Args:
        df (pd.DataFrame): DataFrame con los datos crudos de pingüinos.
        
    Returns:
        DecisionTreeClassifier: Modelo entrenado de Árbol de Decisión.
        
    Note:
        - Utiliza max_depth=5 para evitar overfitting
        - Aplica class_weight='balanced' para manejar desbalance de clases
        - Guarda los resultados en 'models_performance/decision_tree_results.txt'
        - Guarda el modelo en 'models/decision_tree.pkl'
    """
    #X, y, encoders = preprocess_data(df)

    

    X = df.drop(columns=["uuid", "Cover_Type"])
    y = df["Cover_Type"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    model = DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=42#, verbose =  True
    )

    model.fit(X_train, y_train)

    return evaluate_model(
        model, X_test, y_test,
        #ncoders,
        scaler=None,
        model_name="decision_tree"
    )


def train_svm(df):
    """Entrena un modelo de SVM usando OneHotEncoder.
    
    Preprocesa los datos, aplica normalización estándar, divide en conjuntos
    de entrenamiento y prueba, entrena un SVC y evalúa su desempeño.
    
    Args:
        df (pd.DataFrame): DataFrame con los datos crudos de pingüinos.
        
    Returns:
        SVC: Modelo entrenado de SVM.
        
    Note:
        - Utiliza kernel='rbf' para capturar relaciones no lineales
        - Aplica StandardScaler para normalizar las características
        - Aplica class_weight='balanced' para manejar desbalance de clases
        - Guarda los resultados en 'models_performance/svm_results.txt'
        - Guarda el modelo y el scaler en 'models/svm.pkl'
    """
    #X, y, encoders = preprocess_data(df)

    X = df.drop(columns=["uuid", "Cover_Type"])
    y = df["Cover_Type"]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    model = SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    )

    model.fit(X_train, y_train)

    return evaluate_model(
        model, X_test, y_test,
        #encoders,
        scaler=scaler,
        model_name="svm"
    )

def train_knn(df):
    """Entrena un modelo de KNN usando OneHotEncoder.
    
    Preprocesa los datos, aplica normalización estándar, divide en conjuntos
    de entrenamiento y prueba, entrena un KNeighborsClassifier y evalúa su desempeño.
    
    Args:
        df (pd.DataFrame): DataFrame con los datos crudos de pingüinos.
        
    Returns:
        KNeighborsClassifier: Modelo entrenado de KNN.
        
    Note:
        - Utiliza n_neighbors=5 para determinar la clasificación
        - Aplica StandardScaler para normalizar las características
        - Guarda los resultados en 'models_performance/knn_results.txt'
        - Guarda el modelo y el scaler en 'models/knn.pkl'
    """
    #X, y, encoders = preprocess_data(df)


    X = df.drop(columns=["uuid", "Cover_Type"])
    y = df["Cover_Type"]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    model = KNeighborsClassifier(n_neighbors=5)

    model.fit(X_train, y_train)

    return evaluate_model(
        model, X_test, y_test,
        #encoders,
        scaler=scaler,
        model_name="knn"
    )

## Para evaluar los modelos

In [13]:
def evaluate_model(model, X_test, y_test, scaler, model_name):
    """
    Evalúa el desempeño de un modelo entrenado sobre el conjunto de prueba.

    Genera predicciones utilizando el modelo, calcula métricas de evaluación
    (classification report y matriz de confusión), guarda los resultados en
    archivos dentro del directorio `artifacts`, y persiste el modelo entrenado
    para su posterior uso en inferencia.

    Args:
        model: Modelo entrenado (DecisionTreeClassifier, SVC, KNeighborsClassifier, etc.).
        X_test (array-like): Características del conjunto de prueba.
        y_test (array-like): Etiquetas verdaderas del conjunto de prueba.
        scaler: StandardScaler u otro objeto de normalización utilizado durante
            el entrenamiento. Puede ser None si no se utilizó escalamiento.
        model_name (str): Nombre identificador del modelo (por ejemplo:
            'decision_tree', 'svm', 'knn').

    Returns:
        model: El mismo modelo evaluado, para permitir encadenamiento de operaciones.

    Notes:
        - Calcula:
            * Classification Report (precision, recall, f1-score)
            * Confusion Matrix
        - Guarda los resultados en:
            artifacts/{model_name}_results.txt
        - Guarda métricas resumidas en formato JSON:
            artifacts/{model_name}_metrics.json
        - Persiste el modelo utilizando la función `save_model()` en:
            artifacts/{model_name}.pkl
        - El modelo guardado puede ser utilizado posteriormente por servicios
          de inferencia (por ejemplo una API FastAPI).

    Example:
        >>> evaluate_model(model, X_test, y_test, scaler, "decision_tree")
    """

    y_pred = model.predict(X_test)

    report = classification_report(y_test, y_pred)
    matrix = confusion_matrix(y_test, y_pred)

    results_text = f"""
MODEL: {model_name.upper()}

CLASSIFICATION REPORT
{report}

CONFUSION MATRIX
{matrix}
"""

    results_path = artifact_dir / f"{model_name}_results.txt"

    with open(results_path, "w") as f:
        f.write(results_text)

    metrics = {
        "accuracy": (y_pred == y_test).mean()
    }

    metrics_path = artifact_dir / f"{model_name}_metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2))

    print(f"Results saved in: {results_path}")
    print(f"Metrics saved in: {metrics_path}")

    save_model(model, scaler, f"{model_name}.pkl")

    return model

## Para guardar/cargar modelos

In [14]:
# ========================= Funciones de persistencia de modelos =========================

import os
import pickle
import tempfile


def save_model(model, scaler, filename):
    """
    Guarda un modelo entrenado junto con sus componentes de preprocesamiento
    dentro del directorio de artefactos.

    Serializa el modelo y el scaler en un archivo binario utilizando pickle.
    La escritura se realiza primero en un archivo temporal y luego se reemplaza
    el archivo final de forma atómica, evitando que otros procesos (por ejemplo
    una API de inferencia) lean un archivo incompleto durante el guardado.

    Args:
        model: Modelo entrenado (DecisionTreeClassifier, SVC, KNeighborsClassifier, etc.).
        scaler: Objeto de normalización utilizado en el entrenamiento (por ejemplo
            StandardScaler). Puede ser None si el modelo no requiere escalamiento.
        filename (str): Nombre del archivo donde se almacenará el modelo dentro
            del directorio `artifacts`.

    Returns:
        None

    Notes:
        - El archivo final se guarda en `artifacts/{filename}`.
        - Se utiliza escritura atómica para evitar inconsistencias si otro
          proceso intenta leer el archivo durante el guardado.
        - El payload guardado contiene:
            {
                "model": modelo_entrenado,
                "scaler": scaler_utilizado
            }

    Raises:
        OSError: Si ocurre un error durante la escritura o reemplazo del archivo.

    Example:
        >>> save_model(model, scaler, "decision_tree.pkl")
    """

    payload = {
        "model": model,
        "scaler": scaler
    }

    model_path = artifact_dir / filename
    model_path.parent.mkdir(parents=True, exist_ok=True)

    with tempfile.NamedTemporaryFile(delete=False, dir=model_path.parent, suffix=".tmp") as tmp:
        pickle.dump(payload, tmp)
        tmp_path = tmp.name

    os.replace(tmp_path, model_path)

    print(f"Model saved in: {model_path}")


def load_model(filename):
    """Carga un modelo entrenado junto con sus componentes de preprocesamiento.
    
    Deserializa el modelo, los encoders y el scaler desde un archivo pickle
    previamente guardado.
    
    Args:
        filename (str): Ruta y nombre del archivo desde donde cargar el modelo.
        
    Returns:
        tuple: Tupla contiendo:
            - model: Modelo entrenado (DecisionTreeClassifier, SVC o KNeighborsClassifier).
            - encoders (dict): Diccionario con los OneHotEncoders para variables categóricas.
            - scaler: StandardScaler utilizado para normalizar características (puede ser None).
            
    Raises:
        FileNotFoundError: Si el archivo especificado no existe.
        pickle.UnpicklingError: Si el archivo no contiene un objeto pickle válido.
        
    Note:
        - El archivo debe haber sido creado con la función save_model()
        - Los componentes devueltos se pueden usar para realizar predicciones en nuevos datos
    
    Example:
        >>> model, encoders, scaler = load_model('models/my_model.pkl')
    """
    with open(filename, "rb") as f:
        data = pickle.load(f)

    return data["model"], data["encoders"], data["scaler"]

In [15]:
import os
import boto3

artifact_dir = Path('artifacts')
artifact_dir.mkdir(exist_ok=True)

model_path = artifact_dir / 'iris_rf.joblib'
metrics_path = artifact_dir / 'metrics.json'

#pickle.dump(model, model_path)
#metrics_path.write_text(json.dumps({'accuracy': acc}, indent=2))

## Entrenar y guardar modelos

In [16]:
# ========================= Ejecución del script de entrenamiento =========================

if __name__ == "__main__":

    #df = pd.read_csv("dataset_test\dataset_test.csv")

    print("Training Decision Tree...")
    train_decision_tree(df_processed)

    #print("Training SVM...")
    #train_svm(df_processed)

    print("Training KNN...")
    train_knn(df_processed)

    print("All models trained and saved successfully.")

Training Decision Tree...
Results saved in: artifacts/decision_tree_results.txt
Metrics saved in: artifacts/decision_tree_metrics.json
Model saved in: artifacts/decision_tree.pkl
Training KNN...
Results saved in: artifacts/knn_results.txt
Metrics saved in: artifacts/knn_metrics.json
Model saved in: artifacts/knn.pkl
All models trained and saved successfully.


### Subir modelos a minio

In [17]:
endpoint = os.getenv('MINIO_ENDPOINT', 'http://minio:9000')
access_key = os.getenv('AWS_ACCESS_KEY_ID', 'admin')
secret_key = os.getenv('AWS_SECRET_ACCESS_KEY', 'supersecret')
bucket = 'models-bucket'

In [18]:
s3 = boto3.client(
    's3',
    endpoint_url=endpoint,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'),
)

In [19]:
# tree path
model_tree_path = artifact_dir / 'decision_tree.pkl'
metrics_tree_path = artifact_dir / 'decision_tree_metrics.json'

# knn path
model_knn_path = artifact_dir / 'knn.pkl'
metrics_knn_path = artifact_dir / 'knn_metrics.json'

# svm path
model_svm_path = artifact_dir / 'svm.pkl'
metrics_svm_path = artifact_dir / 'svm_metrics.json'

In [20]:
from botocore.client import Config
from botocore.exceptions import ClientError

def ensure_bucket(s3,bucket):

    #s3 = get_s3()

    try:
        s3.head_bucket(Bucket=bucket)
        print("Bucket ya existe")

    except ClientError:
        s3.create_bucket(Bucket=bucket)
        print("Bucket creado")

#ensure_bucket(s3, bucket) # si desea crear el bucket desde aquí lo puede hacer

# load tree model to minio
s3.upload_file(str(model_tree_path), bucket, 'models/decision_tree.pkl')
s3.upload_file(str(metrics_tree_path), bucket, 'metrics/decision_tree_metrics.json')

# load knn model to minio
s3.upload_file(str(model_knn_path), bucket, 'models/knn.pkl')
s3.upload_file(str(metrics_knn_path), bucket, 'metrics/knn_metrics.json')

# load svm model to minio
s3.upload_file(str(model_tree_path), bucket, 'models/svm.pkl')
s3.upload_file(str(metrics_tree_path), bucket, 'metrics/svm_metrics.json')

'Uploaded to MinIO'

Bucket ya existe


'Uploaded to MinIO'